In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')
chat = ChatGoogleGenerativeAI(model = 'gemini-2.0-flash', temperature = 0.7)

In [26]:
from langchain.document_loaders import PyMuPDFLoader

file = 'D:\cs50\gfg tensorflow\Syllabus 3rd Year AIDS V & VI Sem..pdf'
loader = PyMuPDFLoader(file_path = file)
docx = loader.load()

print(docx[0].page_content)

<>:3: SyntaxWarning: invalid escape sequence '\c'
<>:3: SyntaxWarning: invalid escape sequence '\c'
C:\Users\HARDIK JAIN\AppData\Local\Temp\ipykernel_12608\3528847347.py:3: SyntaxWarning: invalid escape sequence '\c'
  file = 'D:\cs50\gfg tensorflow\Syllabus 3rd Year AIDS V & VI Sem..pdf'


Syllabus of  
UNDERGRADUATE DEGREE COURSE 
  
Artificial Intelligence and Data Science 
 
 
 
 
 
 
 
 
Rajasthan Technical University, Kota 
Effective from session: 2022 – 2023


In [27]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"D:\cs50\gfg tensorflow\rich-howl-327614-3cebc91b3316.json"


In [28]:
import google.auth

creds, project = google.auth.default()
print("Authenticated Project ID:", project)


Authenticated Project ID: rich-howl-327614


In [49]:
from vertexai.preview.language_models import TextEmbeddingModel
import vertexai

vertexai.init(project="rich-howl-327614", location="us-central1")  # example: "us-central1"

from langchain.embeddings.base import Embeddings

class GeminiEmbeddings(Embeddings):
    def __init__(self):
        self.model = TextEmbeddingModel.from_pretrained("gemini-embedding-001")

    def embed_documents(self, texts):
        return [e.values for e in self.model.get_embeddings(texts)]

    def embed_query(self, text):
        return self.model.get_embeddings([text])[0].values


In [54]:
from langchain.vectorstores import DocArrayInMemorySearch
import time
from langchain.schema import Document

embeddings = GeminiEmbeddings()
embedded_docs = []
vectors = []
for i, doc in enumerate(docx):
    try:
        text = doc.page_content
        vector = embeddings.embed_query(text)  # One document at a time
        embedded_docs.append(Document(page_content=text, metadata={"page": i + 1}))
        vectors.append(vector)  # ✅ Save vector
        print(f"✅ Embedded page {i + 1}")
        time.sleep(12)  # ⏳ To avoid hitting 5 requests/minute limit
    except Exception as e:
        print(f"❌ Error embedding page {i + 1}: {e}")


print("✅ Embedding completed")

✅ Embedded page 1
✅ Embedded page 2
✅ Embedded page 3
✅ Embedded page 4
✅ Embedded page 5
✅ Embedded page 6
✅ Embedded page 7
✅ Embedded page 8
✅ Embedded page 9
✅ Embedded page 10
✅ Embedded page 11
✅ Embedded page 12
✅ Embedded page 13
✅ Embedded page 14
✅ Embedded page 15
✅ Embedded page 16
✅ Embedded page 17
✅ Embedded page 18
✅ Embedded page 19
✅ Embedded page 20
✅ Embedded page 21
✅ Embedded page 22
✅ Embedded page 23
✅ Embedded page 24
✅ Embedded page 25
✅ Embedded page 26
✅ Embedded page 27
✅ Embedded page 28
✅ Embedded page 29
✅ Embedded page 30
✅ Embedded page 31
✅ Embedding completed


In [83]:
text_embedding_pairs = [(doc.page_content, vector) for doc, vector in zip(embedded_docs, vectors)]

from langchain.vectorstores.faiss import FAISS

db = FAISS.from_embeddings(text_embedding_pairs, embedding=embeddings)


In [ ]:
retriever = db.as_retriever()

from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm = chat, 
    retriever = retriever, 
    return_source_documents = True, 
    chain_type = "stuff", 
    verbose = True
    )





> Entering new RetrievalQA chain...

> Finished chain.
{'query': 'What topics are there in 5th semester?', 'result': 'Based on the provided documents, the following topics are covered in the 5th semester:\n\n*   **Probability and Statistics for Data Science**\n*   **Fundamentals of Blockchain**\n*   **Natural Language Processing (NLP)**', 'source_documents': [Document(id='d1a2d674-1e01-4b07-8781-5d56b92c2872', metadata={}, page_content='RAJASTHAN TECHNICAL UNIVERSITY, KOTA \nSyllabus of 3rd Year B. Tech. (AID) for students admitted in Session 2021-22 onwards. Page 4 \n \n \nSyllabus \nB.Tech.: Artificial Intelligence and Data Science \n5AID5-12: Probability and Statistics for Data Science \nCredit: 2 \nMax. Marks: 100(IA:30, ETE:70) \n2L+0T+0P \nEnd Term Exam: 3 Hours \nObjectives: \n1. To provide advanced statistical background for analysing data and drawing inferences from that \nanalysis. \n2. Predicative Analytics using liner and generalized liner model. \nOutcomes: \nAfter compl

In [107]:
q = "How many total subjects are there in the 5th semester syllabus? Name them all. "
response = qa.invoke({"query":q})
print(response)



> Entering new RetrievalQA chain...

> Finished chain.
{'query': 'How many total subjects are there in the 5th semester syllabus? Name them all. ', 'result': 'Based on the provided syllabus excerpts, there are four subjects listed for the 5th semester:\n\n1.  Fundamentals of Blockchain\n2.  Natural Language Processing (NLP)\n3.  Artificial Neural Network\n4.  Advance Java Lab (this is a 4th semester subject, but included in the provided documents)', 'source_documents': [Document(id='36ea8f1d-89f6-476d-8016-bf834f08883a', metadata={}, page_content='RAJASTHAN TECHNICAL UNIVERSITY, KOTA \nSyllabus of 3rd Year B. Tech. (AID) for students admitted in Session 2021-22 onwards. Page 4 \n \n \nSyllabus \nB.Tech.: Artificial Intelligence and Data Science \n5AID5-11: Fundamentals of Blockchain \nCredit: 2 \nMax. Marks: 100(IA:30, ETE:70) \n2L+0T+0P \nEnd Term Exam: 3 Hours \n \nCourse Objectives: \n1. The students should be able to understand a broad overview of the essential concepts of \nbloc

In [108]:
print(response['result'])

Based on the provided syllabus excerpts, there are four subjects listed for the 5th semester:

1.  Fundamentals of Blockchain
2.  Natural Language Processing (NLP)
3.  Artificial Neural Network
4.  Advance Java Lab (this is a 4th semester subject, but included in the provided documents)
